# PM2.5 ACF(Phase 3 Stream 0)

目的:量化高雄 1287 根竿體 PM2.5 的自相關結構,**決定 Phase 3c 的 lag 預設值**。

Stream 0 驗收條件之一:
> ACF 圖確認 PM2.5 在 lag=24 仍有顯著自相關

另外要檢查:
- lag=1:AR(1) 強度(應接近 1)
- lag=6:6 小時尺度(短時邊界)
- lag=24:日週期
- lag=72:三日邊界(若這裡仍顯著,要考慮加進 lag 預設)
- lag=168:週週期(交通模式)

策略:從 1287 站隨機抽 50 站,**對 train 段(0:1164 小時)** 算 per-station ACF,然後跨站平均。

In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from smart_pole.data.loader import load_pole_hourly
from smart_pole.visualization.plots import setup_chinese_font

PROJECT_ROOT = Path('..').resolve()
setup_chinese_font(PROJECT_ROOT / 'NotoSansCJKtc-Regular.otf')

In [2]:
dataset = load_pole_hourly(
    cache_path=PROJECT_ROOT / 'data' / 'pole_hourly.parquet',
    station_info_path=PROJECT_ROOT / 'data' / 'MOENV_iot_station.csv',
    start='2025-12-04', end='2026-02-05',
)
T, S = dataset.values.shape
train_end = int(T * 0.8)
values_train = dataset.values[:train_end]
print(f'T_train={train_end}, S={S}')

T_train=1164, S=1287


In [3]:
def acf_nan_aware(x: np.ndarray, max_lag: int) -> np.ndarray:
    """以「成對非 NaN」樣本算 Pearson 相關,回傳 length=max_lag+1 的 ACF。
    
    用全段 mean / std 做標準化(經典 ACF 定義),lag=0 結果應 = 1。
    每個 lag 只用 (x[t], x[t+lag]) 同時非 NaN 的 pair。
    """
    x = np.asarray(x, dtype=np.float64)
    finite = np.isfinite(x)
    if finite.sum() < 30:
        return np.full(max_lag + 1, np.nan)
    mean = float(np.nanmean(x))
    var  = float(np.nanvar(x))
    if var == 0.0:
        return np.full(max_lag + 1, np.nan)
    out = np.full(max_lag + 1, np.nan)
    for lag in range(max_lag + 1):
        if lag == 0:
            out[0] = 1.0
            continue
        a = x[:-lag] - mean
        b = x[lag:]  - mean
        m = np.isfinite(a) & np.isfinite(b)
        if m.sum() < 30:
            continue
        out[lag] = float((a[m] * b[m]).mean()) / var
    return out

In [4]:
MAX_LAG = 200       # 至少看到 168 (週) 還要留 buffer
N_STATIONS = 50

rng = np.random.default_rng(0)
sample_idx = rng.choice(S, size=N_STATIONS, replace=False)
acfs = np.full((N_STATIONS, MAX_LAG + 1), np.nan)
for i, sidx in enumerate(sample_idx):
    acfs[i] = acf_nan_aware(values_train[:, sidx], MAX_LAG)

acf_mean = np.nanmean(acfs, axis=0)
acf_std  = np.nanstd(acfs, axis=0, ddof=1)
# 約略 95% confidence band (n=N_STATIONS)
se = acf_std / np.sqrt(N_STATIONS)
print(f'ACF 計算完成 ({N_STATIONS} 站,lag 0–{MAX_LAG})')
print(f'lag=1  ACF = {acf_mean[1]:.3f} ± {se[1]:.3f}')
print(f'lag=6  ACF = {acf_mean[6]:.3f} ± {se[6]:.3f}')
print(f'lag=24 ACF = {acf_mean[24]:.3f} ± {se[24]:.3f}')
print(f'lag=72 ACF = {acf_mean[72]:.3f} ± {se[72]:.3f}')
if MAX_LAG >= 168:
    print(f'lag=168 ACF = {acf_mean[168]:.3f} ± {se[168]:.3f}')

ACF 計算完成 (50 站,lag 0–200)
lag=1  ACF = 0.889 ± 0.007
lag=6  ACF = 0.432 ± 0.011
lag=24 ACF = 0.182 ± 0.009
lag=72 ACF = 0.068 ± 0.010
lag=168 ACF = -0.032 ± 0.008


In [5]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)

ax = axes[0]
lags = np.arange(MAX_LAG + 1)
ax.fill_between(lags, acf_mean - se, acf_mean + se, alpha=0.25, label='±1 SE')
ax.plot(lags, acf_mean, lw=1.2, label='平均 ACF')
# 統計顯著線 (n ≈ 1164 train hr, ±2/√n ≈ ±0.06)
sig_line = 2.0 / np.sqrt(values_train.shape[0])
ax.axhline( sig_line, color='red', linestyle=':', lw=0.8, label=f'±2/√n ≈ ±{sig_line:.3f}')
ax.axhline(-sig_line, color='red', linestyle=':', lw=0.8)
ax.axhline(0, color='k', lw=0.4)
for L, color in [(1, 'C2'), (6, 'C2'), (24, 'C3'), (72, 'C4'), (168, 'C5')]:
    if L <= MAX_LAG:
        ax.axvline(L, color=color, alpha=0.3, lw=0.8)
        ax.text(L, 0.95, f'L={L}', color=color, ha='center', va='top', fontsize=8)
ax.set_xlabel('lag (hr)'); ax.set_ylabel('autocorrelation')
ax.set_title(f'PM2.5 ACF — {N_STATIONS} 隨機站平均(train 段 {values_train.shape[0]}hr)')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, MAX_LAG)

ax = axes[1]
# zoom 在 lag=18-30 看日週期峰
z_start, z_end = 18, 50
lz = lags[z_start:z_end + 1]
ax.fill_between(lz, acf_mean[z_start:z_end + 1] - se[z_start:z_end + 1],
                    acf_mean[z_start:z_end + 1] + se[z_start:z_end + 1], alpha=0.25)
ax.plot(lz, acf_mean[z_start:z_end + 1], 'o-', lw=1.0, markersize=4)
ax.axvline(24, color='C3', alpha=0.5, lw=0.8); ax.text(24, ax.get_ylim()[1], 'L=24', color='C3', va='top')
ax.axhline(sig_line, color='red', linestyle=':', lw=0.8)
ax.set_xlabel('lag (hr)'); ax.set_ylabel('autocorrelation')
ax.set_title('Zoom:lag 18–50,看日週期(24h)峰是否清楚')

fig.tight_layout()
fig.savefig(PROJECT_ROOT / 'results' / 'pm25_acf.png', dpi=120)
plt.show()

In [6]:
# 印 PACF 風格的「lag 1 起降速」表,幫忙挑 lag 預設值
print('lag 1–6:')
for L in range(1, 7):
    sig = '*' if abs(acf_mean[L]) > sig_line else ' '
    print(f'  lag={L:3d}  ACF = {acf_mean[L]:+.3f} {sig}')
print('\n關鍵 lags:')
for L in [6, 12, 24, 48, 72, 96, 120, 168]:
    if L <= MAX_LAG:
        sig = '*' if abs(acf_mean[L]) > sig_line else ' '
        print(f'  lag={L:3d}  ACF = {acf_mean[L]:+.3f} {sig}')

print(f'\n顯著閾值 ±2/√n = ±{sig_line:.3f}')

lag 1–6:
  lag=  1  ACF = +0.889 *
  lag=  2  ACF = +0.772 *
  lag=  3  ACF = +0.669 *
  lag=  4  ACF = +0.578 *
  lag=  5  ACF = +0.499 *
  lag=  6  ACF = +0.432 *

關鍵 lags:
  lag=  6  ACF = +0.432 *
  lag= 12  ACF = +0.208 *
  lag= 24  ACF = +0.182 *
  lag= 48  ACF = +0.142 *
  lag= 72  ACF = +0.068 *
  lag= 96  ACF = +0.080 *
  lag=120  ACF = +0.002  
  lag=168  ACF = -0.032  

顯著閾值 ±2/√n = ±0.059


## 結論模板(實際數字看上面 print)

- lag=1 ACF:**[填數字]** — AR(1) 強度,通常 > 0.95
- lag=24 ACF:**[填數字]** — 日週期峰
- lag=168 ACF:**[填數字]** — 週週期峰

Phase 3c `lags` 預設值取決於這份分析。CLAUDE.md 暫定 `[1, 2, 3, 6, 24]`——
若 lag=168 也顯著(> 2/√n)會考慮加進去,但 168 個 lag column 會炸 K=10 模型的 feature 數。